Design DWV VP3 binders with Mosaic in a Colab environment.

Everything runs **in the notebook cells**. No subprocess, no separate venv.

**How the install works.** Mosaic declares its git dependencies in `[tool.uv.sources]`,
which **pip cannot read** — that is why plain `pip install` kept fetching the unrelated
PyPI `protenix` / `boltz` / `esm`. The real Protenix source is `escalante-bio/protenij`
(with a *j*). Cell 3 ships those 16 git sources already written out as ordinary pinned
`git+https://…@commit` URLs, so ordinary `%pip` installs the right code — straight into
this kernel, so every import works natively in a cell.

**Order:** run cell 1, then 2, then 3. Cell 3 ends by asking you to
**Runtime → Restart session** — that is required, not optional. After restarting, re-run
cell 1 (to redefine the config), then continue from cell 4.

## 1 · Config  *(re-run this after the restart)*

In [ ]:
GROUP = "A_ridge"     # A_ridge = Q325 P326 V384 N386 | B_triad = H277 D294
# B_triad is named after a PUTATIVE catalytic triad (Asp294-His277-Ser278) that did not
# survive scrutiny: no hydrogen-bond geometry between any pair, and the only functional
# assay ever run against it was negative. It is geometrically NON-CATALYTIC. Kept as a
# comparison arm only; no function-blocking claim attaches to it.

MODEL          = "protenix_v2"   # protenix_v2 | protenix_2025 | boltz2
N_DESIGNS      = 30
BINDER_LENGTH  = 80
RECYCLING      = 1               # >1 optimises better but is much slower
SAMPLING_STEPS = 20
ALLOW_CYS      = False           # default: cysteine-free designs

DRIVE_ROOT = "/content/drive/MyDrive/Mosaic"

EPITOPES = {"A_ridge": [325, 326, 384, 386], "B_triad": [277, 294]}
EXPECTED = {"A_ridge": "QPVN", "B_triad": "HD"}
assert GROUP in EPITOPES, f"GROUP must be one of {list(EPITOPES)}"

RUN_NAME = f"5G51_{GROUP}_{MODEL}"
OUT_DIR  = f"{DRIVE_ROOT}/{RUN_NAME}"
TARGET   = f"{OUT_DIR}/5G51_chainA_260_416.pdb"

# 5G51 chain A spans PDB residues 260-416, so index = PDB number - 260.
PDB_OFFSET   = 260
EPITOPE_PDB  = EPITOPES[GROUP]
EPITOPE_IDX  = [r - PDB_OFFSET for r in EPITOPE_PDB]

print(f"group    : {GROUP}  PDB {EPITOPE_PDB}  ->  idx {EPITOPE_IDX}")
print(f"model    : {MODEL}")
print(f"designs  : {N_DESIGNS} x {BINDER_LENGTH} aa, cys_free={not ALLOW_CYS}")
print(f"output   : {OUT_DIR}")

## 2 · GPU + Drive

In [ ]:
import subprocess, os

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu if gpu else "!! NO GPU - Runtime > Change runtime type > GPU, then rerun")

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir ready:", OUT_DIR)

## 3 · Install (~10–20 min, once per session)

Plain `%pip` — the IPython magic, which installs into **this** kernel (that is what
`ModuleNotFoundError: No module named 'mosaic'` was about; a plain shell `pip` can land in
a different interpreter).

The only reason `pip install mosaic` fails on its own is that mosaic declares its git
dependencies in `[tool.uv.sources]`, a table pip cannot read — so it fetches the unrelated
PyPI `protenix` / `boltz` / `esm` instead. The real Protenix source is
`escalante-bio/protenij` (with a *j*). The requirements below already have those 16 git
sources written out as ordinary pinned `git+https://…@commit` URLs, so pip installs the
right code. 238 packages, resolution verified.

In [ ]:
REQUIREMENTS = r"""
ablang==0.3.1
ablang2==0.2.1
absl-py==2.5.0
accelerate==1.14.0
aiofiles==25.1.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
annotated-types==0.8.0
antlr4-python3-runtime==4.9.3
anyio==4.14.2
anywidget==0.11.0
asttokens==3.0.2
attrs==26.1.0
awscrt==0.36.2
biopython==1.84
biotite==1.4.0
biotraj==1.2.2
boltz @ git+https://github.com/escalante-bio/boltz.git@1acc397b6e81f30dc07a80d17a26c38f9c6f6942
boltzgen @ git+https://github.com/escalante-bio/boltzgen.git@c0bbc7ec0dad7f7bc11d28cd83297fb02ddb757a
boto3==1.43.74
botocore==1.43.74
brotli==1.2.0
certifi==2026.7.22
cffi==2.1.1 ; implementation_name == 'pypy' and sys_platform != 'emscripten'
charset-normalizer==3.5.1
chembl-structure-pipeline==1.2.2
click==8.1.7
cloudpathlib==0.24.0
colorama==0.4.6
comm==0.2.3
contourpy==1.3.3
cuequivariance==0.11.1
cuequivariance-ops-cu12==0.11.0
cuequivariance-ops-torch-cu12==0.11.0
cuequivariance-torch==0.11.1
cycler==0.12.1
dm-haiku==0.0.17
dm-tree==0.1.10
dna-features-viewer==3.1.5
docker-pycreds==0.4.0
dockq @ git+https://github.com/nrontsis/DockQ.git@ba4df5adaad7c77fd60851d0b7b05f2b77061ba2
docutils==0.23
edit-distance==1.0.9
einops==0.8.2
einx==0.4.3
equinox==0.13.8
esm @ git+https://github.com/Biohub/esm.git@26b0bc2b771e3e419ea74f445a5f35cc094a1509
esm2quinox==0.2.0
esmjfold2 @ git+https://github.com/escalante-bio/esmjfold2.git@40a33070c4db646152dc374c650f0dedf3c506a6
etils==1.14.0
executing==2.2.1
fairscale==0.4.13
filelock==3.32.3
flax==0.12.8
fonttools==4.63.0
frozendict==2.4.7
frozenlist==1.8.0
fsspec==2026.7.0
func-timeout==4.3.5
gemmi==0.7.5
gitdb==4.0.12
gitpython==3.1.59
h11==0.16.0
hf-xet==1.6.0 ; platform_machine == 'aarch64' or platform_machine == 'amd64' or platform_machine == 'arm64' or platform_machine == 'x86_64'
httpcore==1.0.9
httpx==0.28.1
huggingface-hub==0.36.2
humanize==4.16.0
hydra-core==1.3.2
hydride==1.2.3
idna==3.19
ihm==2.11
ijson==3.5.1
iniconfig==2.3.0
ipymolstar==0.1.0
ipython==9.16.1
ipython-pygments-lexers==1.1.1
ipywidgets==8.1.7
itsdangerous==2.2.0
jablang @ git+https://github.com/escalante-bio/jablang@9ea8157111f8edb5f03818b7c6d417dbd7b53642
jax==0.10.2
jax-cuda12-pjrt==0.10.2
jax-cuda12-plugin==0.10.2
jaxlib==0.10.2
jaxtyping==0.3.11
jedi==0.19.2
jinja2==3.1.6
jmespath==1.1.0
jmp==0.0.4
joblib==1.5.3
joltz @ git+https://github.com/nboyd/joltz@ed0f04257dac85bd4b7bf45521cc280f86d38ded
joltzgen @ git+https://github.com/escalante-bio/joltzgen.git@63925bf2ef5cc033cd4c2bf5c63e638ba338bf24
jopendde @ git+https://github.com/escalante-bio/jopendde.git@8035b69284db6f2bc2302babedcbf5db8c07f4b7
jopenfold3 @ git+https://github.com/escalante-bio/jopenfold3.git@dc8dd6cbefece7398e78e08a9194eac8bb86ce63
jpromera @ git+https://github.com/escalante-bio/jpromera.git@c5b1fe0b22fc2db178ae7bb5480e7f80fe9803e9
jproteina-complexa @ git+https://github.com/escalante-bio/jproteina-complexa.git@018cc136a4bbaa21f66b26db73d122b9a67babf3
jupyterlab-widgets==3.0.17
kalign-python==3.6.0
kiwisolver==1.5.0
lightning-utilities==0.15.3
llvmlite==0.49.0
lmdb==2.3.0
logomaker==0.8.7
loro==1.13.2 ; sys_platform != 'android' and sys_platform != 'emscripten'
marimo==0.24.0
markdown==3.10.3
markdown-it-py==4.2.0
markupsafe==3.0.3
mashumaro==3.14
matplotlib==3.10.5
matplotlib-inline==0.2.2
mdurl==0.1.2
memory-profiler==0.61.0
ml-collections==1.1.0
ml-dtypes==0.6.0
modelcif==1.2
mpmath==1.3.0
msgpack==1.2.1
msgpack-numpy==0.4.8
msgspec==0.21.1
multidict==6.7.1
narwhals==2.24.0
nest-asyncio==1.6.0 ; sys_platform == 'win32'
networkx==3.6.1
numba==0.67.0
numpy==2.4.1
nvidia-cublas-cu12==12.9.2.10
nvidia-cuda-cccl-cu12==12.9.27 ; sys_platform == 'linux'
nvidia-cuda-cupti-cu12==12.9.79 ; sys_platform == 'linux'
nvidia-cuda-nvcc-cu12==12.9.86 ; sys_platform == 'linux'
nvidia-cuda-nvrtc-cu12==12.9.86
nvidia-cuda-runtime-cu12==12.9.79 ; sys_platform == 'linux'
nvidia-cudnn-cu12==9.24.0.43 ; sys_platform == 'linux'
nvidia-cufft-cu12==11.4.1.4 ; sys_platform == 'linux'
nvidia-cusolver-cu12==11.7.3.90
nvidia-cusparse-cu12==12.5.10.65
nvidia-ml-py==13.610.43
nvidia-nccl-cu12==2.31.2 ; sys_platform == 'linux'
nvidia-nvjitlink-cu12==12.9.86
nvidia-nvshmem-cu12==3.7.2 ; sys_platform == 'linux'
omegaconf==2.3.1
opendde @ git+https://github.com/escalante-bio/jopendde.git@8035b69284db6f2bc2302babedcbf5db8c07f4b7#subdirectory=OpenDDE
opt-einsum==3.4.0
optax==0.2.8
optree==0.17.0
orbax-checkpoint==0.12.4
packaging==26.3
pandas==2.3.1
parallelbar==2.5
parso==0.8.7
pdbeccdutils==1.0.0
pexpect==4.9.0 ; sys_platform != 'emscripten' and sys_platform != 'win32'
pillow==12.3.0
platformdirs==4.11.3
pluggy==1.6.0
polars==1.43.2
polars-runtime-32==1.43.2
prometheus-client==0.26.0
prompt-toolkit==3.0.53
propcache==0.5.2
protenix @ git+https://github.com/escalante-bio/protenij@d48a0b0456b582cef10f1bacfb19941a04f7e8c8
protobuf==3.20.2
psutil==7.2.2
psygnal==0.15.1
ptyprocess==0.7.0 ; sys_platform != 'emscripten' and sys_platform != 'win32'
pure-eval==0.2.3
py3dmol==2.5.2
pycparser==3.0 ; implementation_name == 'pypy' and sys_platform != 'emscripten'
pydantic==2.13.4
pydantic-core==2.46.4
pydssp==0.9.1
pygments==2.21.0
pygtrie==2.5.0
pymdown-extensions==11.0.1
pyparsing==3.3.2
pytest==9.1.1
python-dateutil==2.9.0.post0
python-multipart==0.0.32
pytorch-lightning==2.5.0
pytz==2026.3.post1
pyyaml==6.0.2
pyzmq==27.1.0 ; sys_platform != 'emscripten'
rdkit==2026.3.5
regex==2026.7.19
requests==2.32.3
rich==15.0.0
rotary-embedding-torch==0.9.1
ruff==0.16.3
rustworkx==0.18.1
s3transfer==0.19.2
safetensors==0.8.0
scikit-learn==1.7.1
scikit-learn-extra==0.3.0
scipy==1.18.0
sentry-sdk==2.68.0
setproctitle==1.3.7
setuptools==84.0.0
simplejson==4.1.1
six==1.17.0
smmap==5.0.3
stack-data==0.6.3
starlette==1.6.0 ; sys_platform != 'emscripten'
sympy==1.14.0
tabulate==0.10.0
tenacity==9.1.4
tensorstore==0.1.85
threadpoolctl==3.6.0
tinyprot @ git+https://github.com/bjing2016/tinyprot.git@dd2178b650899460223af1a72eb5d85f9ee19619
tokenizers==0.22.2
tomlkit==0.15.1
torch==2.7.1 ; sys_platform == 'darwin'
torch==2.7.1+cpu ; sys_platform != 'darwin'
torchaudio==2.7.1
torchmetrics==1.9.0
torchvision==0.22.1 ; (platform_machine == 'aarch64' and platform_python_implementation == 'CPython' and sys_platform == 'linux') or sys_platform == 'darwin'
torchvision==0.22.1+cpu ; (platform_machine != 'aarch64' and sys_platform == 'linux') or (platform_python_implementation != 'CPython' and sys_platform == 'linux') or (sys_platform != 'darwin' and sys_platform != 'linux')
tqdm==4.67.1
traitlets==5.16.1
transformers @ git+https://github.com/Biohub/transformers.git@ef32577f55da19a4989cd7b22e004dc43a4998cb
treescope==0.1.10
ty==0.0.72
types-requests==2.33.0.20260712
typing-extensions==4.16.0
typing-inspection==0.4.4
tzdata==2026.3
urllib3==2.7.0
uvicorn==0.52.3 ; sys_platform != 'emscripten'
uvloop==0.22.1 ; sys_platform != 'win32'
wadler-lindig==0.1.7
wandb==0.18.7
wcwidth==0.8.2
websockets==17.0.1 ; sys_platform != 'emscripten'
widgetsnbextension==4.0.16
wrapt==2.3.0
yarl==1.24.5
zipp==4.1.0
zstd==1.5.7.2
"""

with open("/content/requirements.txt", "w") as fh:
    fh.write(REQUIREMENTS)
print("requirements written:",
      len([l for l in REQUIREMENTS.strip().splitlines() if l.strip()]), "packages")

In [ ]:
# torch is pinned to the CPU wheel on purpose - torch only loads weights, JAX does the math.
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r /content/requirements.txt

In [ ]:
# mosaic itself, pinned; --no-deps because everything it needs is installed above.
%pip install -q --no-deps git+https://github.com/escalante-bio/mosaic.git@70fec525423f5f87156a1a957b4a4048f9f8e676

print("=" * 62)
print("INSTALL DONE  ->  now do Runtime > Restart session")
print("then re-run cell 1, and continue from cell 4.")
print("=" * 62)

## 4 · Verify + fetch the CCD dictionary  *(after restarting)*

Protenix needs the Chemical Component Dictionary to build residues. It is looked up
at a **fixed path** — `~/.protenix/components.v20240608.cif` — which is *not* affected
by `MOSAIC_CACHE_DIR`. Pointing the cache elsewhere is exactly what produced
`FileNotFoundError: /root/.protenix/components.v20240608.cif`, so we download it to the
path Protenix actually reads.

In [ ]:
import os, jax

print("jax     :", jax.__version__)
print("devices :", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "FATAL: JAX cannot see the GPU"

import mosaic, protenix, boltz
print("mosaic  :", os.path.dirname(mosaic.__file__))
print("protenix:", os.path.dirname(protenix.__file__))

# Boltz2/Boltz1 are CLASSES (capital B); the Protenix* names are FUNCTIONS.
from mosaic.models.protenix import ProtenixV2, Protenix2025, ProtenixBase, ProtenixMini
from mosaic.models.boltz2 import Boltz2
from mosaic.proteinmpnn.mpnn import load_mpnn_sol
from mosaic.optimizers import simplex_APGM
from mosaic.structure_prediction import TargetChain
from mosaic.losses.protein_mpnn import InverseFoldingSequenceRecovery
from mosaic.losses.transformations import NoCys
from mosaic.common import TOKENS
import mosaic.losses.structure_prediction as sp
sp.BinderTargetContact(epitope_idx=[0, 1]); sp.IPSAE_min()

# CCD data -> ~/.protenix (the path protenix.configs.configs_data resolves to).
import protenix.backend as pb
pb._CACHE_DIR = os.path.expanduser("~/.protenix")
pb.download_data()

from protenix.configs.configs_data import data_configs
ccd = data_configs["ccd_components_file"]
assert os.path.exists(ccd), f"CCD still missing at {ccd}"
print(f"CCD ok  : {ccd}  ({os.path.getsize(ccd) / 1e6:.0f} MB)")
print("\nALL IMPORTS OK")

## 5 · Build the target — 5G51 chain A, residues 260–416

**Altloc trap:** 5G51 is 1.45 Å and 21 residues have *no* blank-altloc copy. Filtering to
blank-only silently deletes 263, 270, 271, 278, 291–294, 303, 313, 318, 328, 350, 352,
353, 363, 375, 392, 400, 405, 411 — straight through the epitope. Correct filter is
`altloc in (" ", "A")`.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://files.rcsb.org/download/5G51.pdb", "/content/5g51.pdb")

out, seen = [], set()
for l in open("/content/5g51.pdb"):
    if not l.startswith("ATOM") or l[21] != "A":
        continue
    try:
        rn = int(l[22:26])
    except ValueError:
        continue
    if not (260 <= rn <= 416):
        continue
    if l[16] not in (" ", "A"):        # keep exactly one conformer
        continue
    k = (rn, l[12:16].strip())
    if k in seen:
        continue
    seen.add(k)
    out.append(l[:16] + " " + l[17:])  # blank the altloc column
out.append("TER\nEND\n")

with open(TARGET, "w") as fh:
    fh.writelines(out)

n_ca = sum(1 for l in out if l.startswith("ATOM") and l[12:16] == " CA ")
assert n_ca == 157, f"expected 157 CA, got {n_ca} - altloc filter is wrong"
print(f"target written: {n_ca} residues -> {TARGET}")

## 6 · Load the model and build the loss

Loss weights are the authors' own, from `examples/protenij.py`. The only addition is
`epitope_idx`, which aims `BinderTargetContact` at the chosen hotspots.

Solubility comes from soluble ProteinMPNN (`soluble_v_48_020.pt`) at weight 10.0, plus
cysteine-free design. `NoCys` wraps the **outer** loss and takes an **(N, 19)** simplex,
so the optimizer variable changes shape — `NoCys.sequence()` maps it back to 20 tokens.

First call downloads weights and JIT-compiles: expect **5–10 minutes** of silence.

In [ ]:
import gemmi, jax, numpy as np, jax.numpy as jnp

st = gemmi.read_structure(TARGET)
st.remove_ligands_and_waters()
template_chain = st[0][0]
target_sequence = gemmi.one_letter_code([r.name for r in template_chain])
assert len(target_sequence) == 157, f"target is {len(target_sequence)} residues, expected 157"

# Prove the index mapping before spending any GPU time on it.
observed = "".join(target_sequence[i] for i in EPITOPE_IDX)
print(f"epitope  : PDB {EPITOPE_PDB} -> idx {EPITOPE_IDX} -> '{observed}'")
assert observed == EXPECTED[GROUP], f"maps to '{observed}', expected '{EXPECTED[GROUP]}'"

NO_CYS = not ALLOW_CYS

def make_model(name):
    if name == "protenix_v2":   return ProtenixV2(), "multisample"
    if name == "protenix_2025": return Protenix2025(), "multisample"
    if name == "boltz2":        return Boltz2(), "single"
    raise ValueError(f"unknown model {name!r}")

model, mode = make_model(MODEL)
mpnn = load_mpnn_sol(0.0)

design_features, design_writer = model.binder_features(
    binder_length=BINDER_LENGTH,
    chains=[TargetChain(target_sequence, use_msa=False, template_chain=template_chain)],
)

structure_loss = (
    1.0   * sp.BinderTargetContact(epitope_idx=EPITOPE_IDX)
    + 1.0   * sp.WithinBinderContact()
    + 0.05  * sp.TargetBinderPAE()
    + 0.05  * sp.BinderTargetPAE()
    + 0.025 * sp.IPTMLoss()
    + 0.4   * sp.WithinBinderPAE()
    + 0.025 * sp.pTMEnergy()
    + 0.1   * sp.PLDDTLoss()
    + 10.0  * InverseFoldingSequenceRecovery(mpnn, temp=jnp.array(0.001))
)

if mode == "multisample":
    loss = model.build_multisample_loss(loss=structure_loss, features=design_features,
                                        recycling_steps=RECYCLING, sampling_steps=SAMPLING_STEPS)
else:
    loss = model.build_loss(loss=structure_loss, features=design_features,
                            recycling_steps=RECYCLING, sampling_steps=SAMPLING_STEPS)

if NO_CYS:
    loss = NoCys(loss)

print(f"model + loss ready (cys_free={NO_CYS})")

## 7 · Run the designs

Progress prints live. Every design is written the moment it finishes
(temp file → `fsync` → rename), so an interrupt loses at most one design and re-running
this cell resumes from whatever is already on Drive.

In [ ]:
import json, os, time, glob, traceback

os.makedirs(f"{OUT_DIR}/designs", exist_ok=True)
os.makedirs(f"{OUT_DIR}/pdb", exist_ok=True)

def atomic_write(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        fh.write(text); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)

def optimize(seed):
    """Two-stage schedule from examples/protenij.py: soft solve, then sharpen."""
    n_tok = 19 if NO_CYS else 20
    x = jax.nn.softmax(0.5 * jax.random.gumbel(key=jax.random.key(seed),
                                               shape=(BINDER_LENGTH, n_tok)))
    _, x = simplex_APGM(loss_function=loss, x=x, n_steps=100,
                        stepsize=0.15 * np.sqrt(BINDER_LENGTH),
                        momentum=0.3, scale=1.0, max_gradient_norm=1.0)
    x, _ = simplex_APGM(loss_function=loss, x=jnp.log(x + 1e-5), n_steps=20,
                        stepsize=0.5 * np.sqrt(BINDER_LENGTH),
                        momentum=0.0, scale=1.3, logspace=False, max_gradient_norm=1.0)
    return NoCys.sequence(x) if NO_CYS else x

def metrics_of(pred):
    plddt, pae = np.asarray(pred.plddt), np.asarray(pred.pae)
    b = BINDER_LENGTH
    inter = np.concatenate([pae[:b, b:].ravel(), pae[b:, :b].ravel()])
    return {"iptm": float(np.asarray(pred.iptm)),
            "binder_plddt": float(plddt[:b].mean()),
            "complex_plddt": float(plddt.mean()),
            "interface_pae": float(inter.mean()),
            "min_interface_pae": float(inter.min())}

done = {os.path.basename(p)[:-5] for p in glob.glob(f"{OUT_DIR}/designs/design_*.json")}
print(f"resuming with {len(done)} designs already on Drive\n")

for i in range(N_DESIGNS):
    name = f"design_{i:03d}"
    if name in done:
        continue
    t0 = time.time()
    try:
        pssm = optimize(i)
        ids = np.asarray(pssm).argmax(-1)
        sequence = "".join(TOKENS[int(j)] for j in ids)
        if NO_CYS and "C" in sequence:
            print(f"{name} WARNING: cysteine present despite NoCys")

        pred = model.predict(PSSM=jax.nn.one_hot(ids, 20), features=design_features,
                             writer=design_writer, recycling_steps=4,
                             key=jax.random.key(i))
        m = metrics_of(pred)

        atomic_write(f"{OUT_DIR}/pdb/{name}.pdb", pred.st.make_minimal_pdb())
        atomic_write(f"{OUT_DIR}/designs/{name}.json", json.dumps(
            {"name": name, "group": GROUP, "sequence": sequence, "length": len(sequence),
             "seed": i, "seconds": round(time.time() - t0, 1), **m}, indent=2))

        print(f"{name}  iptm={m['iptm']:.3f}  plddt={m['binder_plddt']:.3f}  "
              f"i_pae={m['interface_pae']:.2f}  ({time.time()-t0:.0f}s)\n  {sequence}",
              flush=True)
    except Exception:
        print(f"{name} FAILED:\n{traceback.format_exc()}", flush=True)

print("\nDONE")

## 8 · Results

In [ ]:
import json, glob, csv

rows = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT_DIR}/designs/*.json"))]
rows.sort(key=lambda r: -r["iptm"])
print(f"{len(rows)} designs\n")

cols = ["name", "sequence", "length", "iptm", "binder_plddt", "complex_plddt",
        "interface_pae", "min_interface_pae", "seed", "seconds"]
with open(f"{OUT_DIR}/summary.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
    w.writeheader(); w.writerows(rows)

for r in rows[:10]:
    print(f"{r['name']}  iptm={r['iptm']:.3f}  plddt={r['binder_plddt']:.3f}  "
          f"i_pae={r['interface_pae']:.2f}\n  {r['sequence']}")
print(f"\nwrote {OUT_DIR}/summary.csv")

## Notes

**Ranking.** `iptm` is recorded for convenience but on your own benchmark it barely
separates binders from non-binders (AUC ≈ 0.5). Rank on **min_ipSAE** — `sp.IPSAE_min`
exists in mosaic and is verified importable in cell 4, but is not yet wired into the
scoring here. Treat this ordering as provisional.

**Both epitope groups.** Run `A_ridge` first. Then set `GROUP = "B_triad"` in cell 1 and (note: the triad B_triad is named for is putative and geometrically non-catalytic -- see cell 1)
re-run cells 1, 5, 6, 7 — separate output folders, so they never collide. Sequentially,
not in parallel: two jobs on one GPU will fight for VRAM.

**If the session disconnects,** re-run cells 1, 2, 4, 5, 6, 7. Cell 3 only needs redoing
if the VM was torn down. Cell 7 skips every design already on Drive.

**`Boltz2` is a class, not a function** — `from mosaic.models.boltz2 import Boltz2`,
capital B. Lowercase `boltz2` is the *module*. The `Protenix*` entry points are functions
returning a configured `Protenix`; both are called the same way.

A standalone CLI version for detached runs is at `mosaic/scripts/mosaic_5g51_design.py`.